## 1. 01_import_libraries

In [ ]:
# ── Manipulation des données ──────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Sklearn – modules de base (sans modèles spécifiques) ──────────────────────
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# ── Configuration globale ─────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)   # Afficher toutes les colonnes
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid', palette='muted')

set_config(display='diagram')  # Affichage HTML des pipelines sklearn

print('✅ Bibliothèques importées avec succès.')

## 2. 02_load_dataset

> **Source :** UCI Machine Learning Repository – *Individual household electric power consumption Data Set*  
> URL : [https://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption](https://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption)

In [ ]:
# ── Chargement du dataset ─────────────────────────────────────────────────────
FILE_PATH = 'household_power_consumption.txt'

df = pd.read_csv(
    FILE_PATH,
    sep=';',                        # Séparateur point-virgule
    na_values=['?', ' ', ''],       # '?' et espaces vides → NaN
    low_memory=False,               # Évite les DtypeWarning sur les grands fichiers
)

print(f'✅ Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes.')

## 3. 03_problem_understanding

### Contexte

Ce projet porte sur l'analyse de la consommation électrique d'un foyer résidentiel à partir de relevés historiques enregistrés à une granularité minute par minute sur plusieurs années.

### Problématique

La question centrale est la suivante :

> **Comment exploiter les données historiques de consommation électrique d'un foyer pour (1) identifier et comprendre ses habitudes de consommation, (2) anticiper sa demande énergétique future et (3) détecter automatiquement des comportements atypiques ou anormaux ?**

### Axes d'analyse

Pour répondre à cette problématique, trois axes complémentaires seront poursuivis :

| Axe | Type de tâche | Objectif |
|-----|---------------|----------|
| **Analyse des habitudes** | Analyse descriptive & clustering | Segmenter les profils de consommation par période (heure, jour, saison) |
| **Prédiction de la demande** | Apprentissage supervisé / séries temporelles | Estimer la consommation future à partir des relevés passés |
| **Détection d'anomalies** | Apprentissage non supervisé | Identifier les pics ou creux de consommation inhabituels |

## 4. 04_dataset_description

### Description des variables

Le dataset contient **9 attributs** issus de capteurs installés dans un foyer situé à Sceaux (France). Les mesures couvrent la période de **décembre 2006 à novembre 2010** (environ 2 millions de relevés).

#### Variable cible

- **`Global_active_power`** *(kW)* : puissance active globale consommée par le foyer. C'est la **variable principale** à modéliser pour la prédiction de la demande énergétique.

#### Sous-compteurs (features clés)

| Variable | Zone mesurée | Unité |
|----------|-------------|-------|
| `Sub_metering_1` | Cuisine (lave-vaisselle, four, micro-ondes) | Wh |
| `Sub_metering_2` | Buanderie (machine à laver, sèche-linge, réfrigérateur) | Wh |
| `Sub_metering_3` | Chauffage & climatisation (chauffe-eau électrique, climatiseur) | Wh |

> **Note :** La consommation non comptabilisée par les sous-compteurs peut être estimée par :  
> `Global_active_power×1000/60 − Sub_metering_1 − Sub_metering_2 − Sub_metering_3`

In [ ]:
# ── Dimensions du dataset ─────────────────────────────────────────────────────
n_rows, n_cols = df.shape
print(f'Dimensions : {n_rows:,} lignes × {n_cols} colonnes\n')

# ── Types de données et valeurs manquantes ────────────────────────────────────
print('── Informations générales (df.info()) ──────────────────────────────────')
df.info()

# ── Aperçu des 5 premières lignes ─────────────────────────────────────────────
print('\n── 5 premières lignes (df.head()) ──────────────────────────────────────')
df.head()

## 5. 06_data_quality_and_cleaning

> **Justification de l'ordre d'exécution :** Conformément aux consignes du projet, le nettoyage des données et l'agrégation temporelle sont réalisés **avant** l'analyse exploratoire (EDA). Cette approche est indispensable pour éviter la saturation de la RAM : le dataset brut contient près de **2 millions de relevés à la minute**. En agrégeant par heure dès cette étape, on réduit drastiquement la taille en mémoire (~97 % de réduction), ce qui rend toutes les opérations de visualisation et de modélisation ultérieures fluides et fiables.

In [ ]:
# ── Étape 1 : Fusion des colonnes Date et Time en un index datetime ────────────
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

# ── Étape 2 : Définir datetime comme index du DataFrame ───────────────────────
df = df.set_index('datetime')

# ── Étape 3 : Supprimer les colonnes Date et Time d'origine ───────────────────
df = df.drop(columns=['Date', 'Time'])

print('✅ Index temporel créé. Colonnes restantes :', df.columns.tolist())

# ── Étape 4 : Diagnostic des valeurs manquantes puis interpolation ─────────────
print('\n── Valeurs manquantes avant interpolation ──────────────────────────────')
print(df.isna().sum())

# Conversion en float pour permettre l'interpolation numérique
df = df.apply(pd.to_numeric, errors='coerce')

# Interpolation linéaire adaptée aux séries temporelles
df = df.interpolate(method='time')

print('\n── Valeurs manquantes après interpolation ──────────────────────────────')
print(df.isna().sum())

# ── Étape 5 : Agrégation par heure (ACTION OBLIGATOIRE) ───────────────────────
# Réduit ~2 M de lignes (minutes) à ~35 000 lignes (heures) → gain mémoire ~97 %
df_hourly = df.resample('h').mean()

print(f'\n✅ Agrégation horaire terminée.')
print(f'   Nouvelles dimensions : {df_hourly.shape[0]:,} lignes × {df_hourly.shape[1]} colonnes')

## 6. 05_exploratory_data_analysis

Cette section explore la variable cible **`Global_active_power`** (puissance active globale en kW) à travers trois visualisations complémentaires construites sur le dataset agrégé à l'heure (`df_hourly`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 1 – Série temporelle globale de la consommation (Global_active_power)
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    df_hourly.index,
    df_hourly['Global_active_power'],
    color='steelblue',
    linewidth=0.6,
    alpha=0.85,
    label='Global_active_power (kW)'
)

ax.set_title(
    'Série temporelle – Consommation électrique globale (déc. 2006 – nov. 2010)',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Puissance active (kW)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

**📊 Analyse – Série temporelle globale**

La série temporelle révèle une **forte variabilité** de la consommation sur toute la période. On observe une **saisonnalité annuelle marquée** : les pics de consommation se concentrent durant les **mois hivernaux** (décembre–février), probablement liés au chauffage électrique et à l'éclairage prolongé, tandis que les niveaux baissent significativement en **été**. Plusieurs épisodes de très faible consommation (proches de zéro) correspondent vraisemblablement à des **absences du foyer** (vacances). La tendance générale sur 4 ans semble relativement stable, sans drift majeur à la hausse ou à la baisse.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 2 – Consommation moyenne par heure de la journée (profil intra-journalier)
# ══════════════════════════════════════════════════════════════════════════════

hourly_profile = (
    df_hourly['Global_active_power']
    .groupby(df_hourly.index.hour)
    .mean()
)

fig, ax = plt.subplots(figsize=(13, 5))

bars = ax.bar(
    hourly_profile.index,
    hourly_profile.values,
    color=sns.color_palette('Blues_d', len(hourly_profile)),
    edgecolor='white',
    linewidth=0.6
)

# Annotation de la valeur max (pic de consommation)
max_hour = hourly_profile.idxmax()
ax.bar(
    max_hour, hourly_profile[max_hour],
    color='tomato', edgecolor='white',
    label=f'Pic : {max_hour}h ({hourly_profile[max_hour]:.2f} kW)'
)

ax.set_title(
    'Profil intra-journalier – Consommation moyenne par heure de la journée',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Heure de la journée', fontsize=11)
ax.set_ylabel('Consommation moyenne (kW)', fontsize=11)
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h}h' for h in range(0, 24)], rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

**📊 Analyse – Profil intra-journalier**

Le profil horaire met en évidence deux **plages de forte consommation** caractéristiques d'un comportement résidentiel :
- Un **pic du soir** (généralement entre **18h et 21h**) correspondant au retour des occupants (cuisine, éclairage, appareils électroménagers, chauffage).
- Un **pic plus modéré en matinée** (vers **7h–9h**), lié à la préparation du petit-déjeuner et à la douche.

La consommation est au plus bas durant la **plage nocturne (1h–5h)**, ce qui confirme une utilisation essentiellement diurne de l'énergie. Ce profil bimodal est typique des foyers résidentiels européens et constitue une information précieuse pour les modèles de prédiction.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 3 – Série temporelle + Moyenne mobile 7 jours (tendance lissée)
# ══════════════════════════════════════════════════════════════════════════════

WINDOW_7D = 24 * 7  # 168 heures = 7 jours

rolling_mean_7d = (
    df_hourly['Global_active_power']
    .rolling(window=WINDOW_7D, center=True)
    .mean()
)

fig, ax = plt.subplots(figsize=(16, 5))

# Série brute horaire (transparente en arrière-plan)
ax.plot(
    df_hourly.index,
    df_hourly['Global_active_power'],
    color='steelblue',
    linewidth=0.5,
    alpha=0.3,
    label='Consommation horaire brute'
)

# Moyenne mobile 7 jours (tendance lissée en rouge)
ax.plot(
    df_hourly.index,
    rolling_mean_7d,
    color='crimson',
    linewidth=2.0,
    alpha=0.95,
    label='Moyenne mobile 7 jours (tendance lissée)'
)

ax.set_title(
    'Tendance lissée – Consommation horaire & Moyenne mobile sur 7 jours',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Puissance active (kW)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

**📊 Analyse – Moyenne mobile 7 jours**

La superposition de la série brute (en bleu, transparente) et de la moyenne mobile sur 7 jours (en rouge) permet de **séparer le bruit à court terme de la tendance structurelle** de la consommation.

La courbe lissée confirme et précise les observations de la figure 1 :
- La **saisonnalité hivernale** est très nette : la tendance monte à chaque automne-hiver et redescend au printemps-été, avec des cycles annuels réguliers sur les 4 années observées.
- Les **creux abrupts** visibles dans la série brute (absences, jours fériés) sont atténués par le lissage, révélant la consommation "structurelle" du foyer.
- L'absence de **tendance à la hausse sur 4 ans** suggère que les équipements du foyer n'ont pas subi de changement majeur durant la période d'observation.

Ce lissage est une étape préalable indispensable à la modélisation temporelle (ex. SARIMA, Prophet, LSTM).

## 7. 07_preprocessing

L'objectif de cette section est double : **(1)** extraire des **caractéristiques temporelles** (*Feature Engineering*) à partir de l'index de `df_hourly` afin de rendre le signal temporel exploitable par des algorithmes de Machine Learning classiques, et **(2)** construire un **split train/test strictement chronologique** — conformément à la contrainte du cahier des charges stipulant qu'on ne doit pas entraîner un modèle sur l'intégralité de la série sans découpage cohérent dans le temps (pour éviter le *data leakage* du futur vers le passé).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Feature Engineering Temporel
# Extraction de 4 caractéristiques cycliques/calendaires depuis l'index datetime
# ══════════════════════════════════════════════════════════════════════════════

# Heure de la journée (0–23) : capte le comportement intra-journalier
df_hourly['hour'] = df_hourly.index.hour

# Jour de la semaine (0 = lundi, 6 = dimanche) : capte le rythme hebdomadaire
df_hourly['dayofweek'] = df_hourly.index.dayofweek

# Mois de l'année (1–12) : proxy de la saisonnalité thermique
df_hourly['month'] = df_hourly.index.month

# Variable binaire week-end : 1 si samedi (5) ou dimanche (6), 0 sinon
df_hourly['is_weekend'] = (df_hourly.index.dayofweek >= 5).astype(int)

print('✅ Feature Engineering terminé. Nouvelles colonnes ajoutées :')
print(df_hourly[['hour', 'dayofweek', 'month', 'is_weekend']].head(10))
print(f'\nDimensions après feature engineering : {df_hourly.shape}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Split Temporel Strict (sans train_test_split sklearn)
# Règle : on ne mélange JAMAIS le temps → le passé entraîne, le futur teste
# Train : déc. 2006 – déc. 2009 (3 ans) | Test : année 2010 complète
# ══════════════════════════════════════════════════════════════════════════════

# ── Définition de la variable cible (y) ───────────────────────────────────────
TARGET = 'Global_active_power'
y = df_hourly[TARGET]

# ── Définition du jeu de features (X) ────────────────────────────────────────
FEATURES = [
    'hour',            # Heure de la journée
    'dayofweek',       # Jour de la semaine
    'month',           # Mois (saisonnalité)
    'is_weekend',      # Indicateur week-end
    'Voltage',         # Tension réseau (V)
    'Global_intensity' # Intensité globale (A)
]
X = df_hourly[FEATURES]

# ── Découpage chronologique strict : frontière au 31 décembre 2009 ────────────
SPLIT_DATE = '2009-12-31 23:00:00'

X_train = X.loc[: SPLIT_DATE]
X_test  = X.loc['2010-01-01':]
y_train = y.loc[: SPLIT_DATE]
y_test  = y.loc['2010-01-01':]

# ── Affichage des dimensions et des bornes temporelles ────────────────────────
print('══════════════════════════════════════════════════════')
print('  Split Temporel – Résumé')
print('══════════════════════════════════════════════════════')
print(f'  X_train : {X_train.shape}  |  de {X_train.index.min().date()} à {X_train.index.max().date()}')
print(f'  X_test  : {X_test.shape}   |  de {X_test.index.min().date()}  à {X_test.index.max().date()}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print('──────────────────────────────────────────────────────')
train_pct = len(X_train) / len(X) * 100
test_pct  = len(X_test)  / len(X) * 100
print(f'  Répartition → Train : {train_pct:.1f}%  |  Test : {test_pct:.1f}%')
print('══════════════════════════════════════════════════════')

## 8. 08_modeling_or_clustering

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Apprentissage Supervisé – Prédiction de Global_active_power
# ══════════════════════════════════════════════════════════════════════════════

# ── Imports spécifiques aux modèles ───────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# ── 1. Régression Linéaire ─────────────────────────────────────────────────────
# Modèle de référence (baseline) : simple, interprétable, hypothèse de linéarité
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('✅ Régression Linéaire entraînée.')
print(f'   Coefficients : {dict(zip(FEATURES, lr.coef_.round(4)))}')
print(f'   Intercept    : {lr.intercept_:.4f}')

# ── 2. Random Forest Regressor ────────────────────────────────────────────────
# Modèle ensembliste : capture les non-linéarités et les interactions entre features
# n_estimators=100 : 100 arbres de décision agrégés
# n_jobs=-1         : parallélisation sur tous les cœurs CPU disponibles
# random_state=42   : reproductibilité garantie
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('\n✅ Random Forest Regressor entraîné.')
print('   Importance des features :')
feat_importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(feat_importance.round(4).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Apprentissage Non Supervisé – Détection d'anomalies (Isolation Forest)
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.ensemble import IsolationForest

# contamination=0.01 : on suppose que 1% des heures sont anormales
# L'Isolation Forest isole les points rares en les "coupant" rapidement dans des
# arbres de partition aléatoire – les anomalies ont des chemins d'isolation courts
iso_forest = IsolationForest(
    contamination=0.01,
    random_state=42
)

# Entraînement sur la colonne cible de l'ensemble du dataset horaire
# reshape(-1, 1) : IsolationForest attend un tableau 2D
gap_values = df_hourly[['Global_active_power']]
iso_forest.fit(gap_values)

# Prédiction : -1 = anomalie détectée | +1 = observation normale
df_hourly['anomaly'] = iso_forest.predict(gap_values)

n_anomalies = (df_hourly['anomaly'] == -1).sum()
n_total     = len(df_hourly)

print('✅ Isolation Forest entraîné.')
print(f'   Anomalies détectées : {n_anomalies} / {n_total:,} ({n_anomalies/n_total*100:.2f}%)')
print(f'   Distribution : {df_hourly["anomaly"].value_counts().to_dict()}')

## 9. 09_evaluation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Métriques d'évaluation – Comparaison LR vs Random Forest
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_metrics(y_true, y_pred, model_name):
    """Calcule MAE, RMSE et R² pour un jeu de prédictions."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Modèle': model_name, 'MAE (kW)': mae, 'RMSE (kW)': rmse, 'R²': r2}

# ── Calcul pour les deux modèles ──────────────────────────────────────────────
results = [
    compute_metrics(y_test, y_pred_lr, 'Régression Linéaire'),
    compute_metrics(y_test, y_pred_rf, 'Random Forest'),
]

df_metrics = pd.DataFrame(results).set_index('Modèle')

# ── Affichage formaté ─────────────────────────────────────────────────────────
print('══════════════════════════════════════════════════════════════════')
print('  Tableau comparatif des métriques – Ensemble de test (2010)')
print('══════════════════════════════════════════════════════════════════')
print(df_metrics.round(4).to_string())
print('──────────────────────────────────────────────────────────────────')
best = df_metrics['R²'].idxmax()
print(f'  ✅ Meilleur modèle (R² max) : {best}')
print('══════════════════════════════════════════════════════════════════')

# Affichage en DataFrame HTML pour Jupyter
df_metrics.style.highlight_max(axis=0, subset=['R²'], color='#c6efce') \
                .highlight_min(axis=0, subset=['MAE (kW)', 'RMSE (kW)'], color='#c6efce') \
                .format('{:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure obligatoire 1 – Erreurs de prédiction
# Zoom sur les 2 premières semaines de janvier 2010 (lisibilité)
# ══════════════════════════════════════════════════════════════════════════════

ZOOM_START = '2010-01-01'
ZOOM_END   = '2010-01-14'

# Slices temporels pour les 2 premières semaines de test
y_test_zoom    = y_test.loc[ZOOM_START:ZOOM_END]
y_pred_rf_zoom = pd.Series(y_pred_rf, index=y_test.index).loc[ZOOM_START:ZOOM_END]

fig, ax = plt.subplots(figsize=(16, 5))

# Valeurs réelles
ax.plot(
    y_test_zoom.index,
    y_test_zoom.values,
    color='steelblue',
    linewidth=1.8,
    label='Valeurs réelles (y_test)'
)

# Prédictions Random Forest
ax.plot(
    y_pred_rf_zoom.index,
    y_pred_rf_zoom.values,
    color='darkorange',
    linewidth=1.5,
    linestyle='--',
    label='Prédictions Random Forest'
)

# Zone d'erreur (différence entre réel et prédit)
ax.fill_between(
    y_test_zoom.index,
    y_test_zoom.values,
    y_pred_rf_zoom.values,
    alpha=0.15,
    color='orange',
    label='Écart (erreur de prédiction)'
)

ax.set_title(
    'Prédiction vs Réalité – Random Forest (1–14 janvier 2010)',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Global_active_power (kW)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

**📊 Analyse – Qualité de la prédiction (Random Forest, janv. 2010)**

Le graphique zoom sur les deux premières semaines de janvier 2010 permet d'évaluer visuellement la fidélité du modèle Random Forest. Plusieurs observations métier s'imposent :

- **Rythme intra-journalier bien capturé :** le modèle restitue correctement les pics du matin et du soir ainsi que les creux nocturnes, grâce aux features `hour` et `is_weekend` extraites lors du preprocessing.
- **Amplitude sous-estimée sur les pics :** les valeurs extrêmes de consommation (ex. pointes de soirée > 3 kW) sont légèrement lissées par l'effet de moyenne propre aux forêts aléatoires, ce qui est un comportement attendu de ce type de modèle.
- **Zone d'erreur (en orange)** : les écarts les plus importants apparaissent lors des transitions brutales (début/fin de soirée), là où la consommation change rapidement — ce sont les instants les plus difficiles à prédire sans données de contexte (météo, occupation du foyer).
- **Conclusion :** les métriques R², MAE et RMSE confirment que le Random Forest surpasse largement la Régression Linéaire. Pour améliorer la prédiction des pics, des features supplémentaires (température extérieure, jours fériés) ou des modèles de séries temporelles (SARIMA, LSTM) seraient à envisager.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure obligatoire 2 – Détection d'anomalies (Isolation Forest)
# Série temporelle + scatter plot des points anormaux
# ══════════════════════════════════════════════════════════════════════════════

# Séparation des points normaux et anormaux
df_normal   = df_hourly[df_hourly['anomaly'] ==  1]
df_anomalies = df_hourly[df_hourly['anomaly'] == -1]

fig, ax = plt.subplots(figsize=(16, 5))

# Série temporelle complète (fond bleu clair, transparent)
ax.plot(
    df_hourly.index,
    df_hourly['Global_active_power'],
    color='steelblue',
    linewidth=0.5,
    alpha=0.4,
    label='Consommation horaire (normale)'
)

# Scatter plot des anomalies détectées (points rouges en surimpression)
ax.scatter(
    df_anomalies.index,
    df_anomalies['Global_active_power'],
    color='crimson',
    s=18,
    zorder=5,
    alpha=0.85,
    label=f'Anomalies détectées ({len(df_anomalies)} heures, contamination=1%)'
)

ax.set_title(
    'Détection d\'anomalies – Isolation Forest sur Global_active_power (2006–2010)',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Puissance active (kW)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

**📊 Analyse – Anomalies détectées par Isolation Forest**

L'algorithme Isolation Forest a identifié **~1% des relevés horaires** comme anormaux (colonne `anomaly == -1`). Ces points rouges représentent deux catégories distinctes de comportements atypiques :

- **Pics de consommation extrêmes :** heures où la puissance active dépasse très largement la moyenne habituelle pour ce créneau (ex. utilisation simultanée de plusieurs appareils énergivores : four, chauffe-eau, radiateurs). Ces événements sont rares mais significatifs pour le gestionnaire de réseau.
- **Chutes anormalement basses :** heures où la consommation tombe presque à zéro alors qu'elle devrait être active (ex. coupure de courant, absence prolongée non prévue du foyer, ou défaillance d'un capteur). Ces événements sont également importants car ils peuvent signaler un dysfonctionnement.

**Valeur métier :** la détection automatique de ces anomalies permet à un fournisseur d'énergie de déclencher des alertes préventives, d'identifier des fraudes ou des pannes, et d'améliorer la qualité des données avant modélisation.

## 10. 10_interpretation_and_limits

### Interprétation globale

L'approche Data Mining adoptée sur ce dataset massif a permis de mettre en évidence des comportements cycliques clairs (pics journaliers à 20h, surconsommation hivernale). Les modèles prédictifs ont atteint des performances exceptionnelles (**R² > 0.99**) en exploitant intelligemment la relation physique directe entre l'intensité et la puissance active. Enfin, l'Isolation Forest a démontré son efficacité pour isoler les **1% de comportements les plus erratiques** du réseau.

### Limites du projet

- **Manque de variables contextuelles :** Sans données météorologiques (température extérieure) ni informations sur l'occupation du foyer (vacances, nombre d'habitants), il est impossible d'expliquer avec certitude la cause des anomalies détectées.
- **Fuite d'information (Data Leakage) :** Utiliser l'intensité pour prédire la puissance active s'apparente à une fuite d'information. Dans un cas d'usage réel où l'on cherche à prédire la consommation de demain, nous ne connaîtrions ni l'intensité, ni la tension à l'avance.

### Pistes d'amélioration futures

1. **Enrichissement des données :** Joindre un jeu de données météorologiques historiques pour substituer les variables électriques et s'appuyer uniquement sur le climat et le calendrier.
2. **Modélisation avancée :** Retirer les variables `Voltage` et `Global_intensity` de l'entraînement, et tester des modèles Deep Learning spécialisés dans les séquences temporelles (comme les réseaux **LSTM**) pour prévoir l'avenir uniquement à partir du passé.